In [28]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Define the physical line shape profiles
def lorentzian(x):
    return 1.0 / (1.0 + 4.0 * x**2)

def gaussian(x):
    return np.exp(-4.0 * np.log(2.0) * x**2)

def calculate_signals(freq_arr, nu_0, delta_nu, nu_dev, shape, contrast=0.1, rate=1.0):
    if shape == 'Lorentzian':
        static_I = rate * (1 - contrast * lorentzian((freq_arr - nu_0) / delta_nu))
    else:
        static_I = rate * (1 - contrast * gaussian((freq_arr - nu_0) / delta_nu))

    theta = np.linspace(0, 2 * np.pi, 128, endpoint=False)
    S_E = np.zeros_like(freq_arr)
    for i, nc in enumerate(freq_arr):
        nu_m = nc + nu_dev * np.sin(theta)
        if shape == 'Lorentzian':
            I_t = rate * (1 - contrast * lorentzian((nu_m - nu_0) / delta_nu))
        else:
            I_t = rate * (1 - contrast * gaussian((nu_m - nu_0) / delta_nu))

        S_E[i] = (1.0 / np.pi) * np.trapz(I_t * np.sin(theta), theta)
    return static_I, S_E

def plot_fm_odmr(nu_0=2870.0, nu_dev=3.46, delta_nu=6.0, contrast=0.1, shape='Lorentzian'):
    freq_arr = np.linspace(2860, 2880, 1000)
    
    static_I, S_E = calculate_signals(freq_arr, nu_0, delta_nu, nu_dev, shape, contrast=contrast)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(freq_arr, static_I, 'k-', linewidth=2, label='Static ODMR Profile')
    ax1.axvspan(nu_0 - nu_dev, nu_0 + nu_dev, color='green', alpha=0.2, label='Modulation Sweep Region')
    ax1.axvline(nu_0, color='green', linestyle=':', label='Resonant Center ν₀')
    ax1.set_title('Static ODMR Dip & Modulation Region')
    ax1.set_xlabel('Microwave Frequency (MHz)')
    ax1.set_ylabel('Fluorescence Intensity (arb. units)')
    ax1.set_xlim(np.min(freq_arr), np.max(freq_arr))
    ax1.legend(loc='lower right')
    ax1.grid(True, linestyle=':', alpha=0.7)

    ax2.plot(freq_arr, S_E, 'b-', linewidth=2, label='Error Signal S_E')
    ax2.axvline(nu_0, color='gray', linestyle='--', label='Zero-Crossing Target')
    ax2.axhline(0, color='gray', linestyle='--')
    center_idx = np.argmin(np.abs(freq_arr - nu_0))
    slope = (S_E[center_idx+1] - S_E[center_idx-1]) / (freq_arr[center_idx+1] - freq_arr[center_idx-1])
    tangent_nu = np.linspace(nu_0 - 3, nu_0 + 3, 20)
    tangent_SE = slope * (tangent_nu - nu_0)
    ax2.plot(tangent_nu, tangent_SE, color='orange', linestyle='--', linewidth=2, label=f'Slope |D| = {abs(slope):.5f}')
    ax2.set_title('Lock-In Error Signal (S_E)')
    ax2.set_xlabel('Tracking Frequency ν_c (MHz)')
    ax2.set_ylabel('Demodulated Amplitude')
    ax2.set_xlim(np.min(freq_arr), np.max(freq_arr))
    ax2.set_ylim(-0.06, 0.06)
    ax2.legend(loc='upper right')
    ax2.grid(True, linestyle=':', alpha=0.7)

    plt.tight_layout()
    plt.show()

# Shared wide layout for sliders
long_slider_layout = widgets.Layout(width='800px')

# Build the interactive widget
w = widgets.interactive(
    plot_fm_odmr,
    nu_0=widgets.FloatSlider(value=2870.0, min=2860.0, max=2880.0, step=0.1, description='Resonant Freq ν₀ (MHz)', continuous_update=False, style={'description_width': 'initial'}, layout=long_slider_layout),
    nu_dev=widgets.FloatSlider(value=1.0, min=0.01, max=10.0, step=0.01, description='Modulation Depth νdev (MHz)', continuous_update=False, style={'description_width': 'initial'}, layout=long_slider_layout),
    delta_nu=widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.01, description='Linewidth Δν (MHz)', continuous_update=False, style={'description_width': 'initial'}, layout=long_slider_layout),
    contrast=widgets.FloatSlider(value=0.05, min=0.01, max=0.1, step=0.001, description='Contrast C', continuous_update=False, style={'description_width': 'initial'}, layout=long_slider_layout),
    shape=widgets.Dropdown(options=['Lorentzian', 'Gaussian'], value='Lorentzian', description='Line Shape', style={'description_width': 'initial'}),
)

# Display: plot output first, then controls below
display(widgets.VBox([w.children[-1], widgets.VBox(list(w.children[:-1]))]))